Importing libraries.

In [21]:
import pandas as pd
import numpy as np
import requests
import re
import os
import time
from lxml import html
from bs4 import BeautifulSoup
from tqdm import tqdm

import selenium
from webdriver_manager.chrome import ChromeDriverManager

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC 
from selenium.webdriver.common.by import By 


Let's define the page we will be scraping.

In [22]:
url = "https://nbp.pl/kategoria/aktualnosci/"

Scraping as a static page.

In [23]:
response = requests.get(url)
print(response.text)

<html style="height:100%"><head><META NAME="ROBOTS" CONTENT="NOINDEX, NOFOLLOW"><meta name="format-detection" content="telephone=no"><meta name="viewport" content="initial-scale=1.0"><meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1"><script src="/Of-slaine-but-secrewes-the-Lye-my-lean-an-of-the" async></script><script type="text/javascript">if (sessionStorage) { sessionStorage.setItem('distil_referrer', document.referrer); }</script></head><body style="margin:0px;height:100%"><iframe id="main-iframe" src="/_Incapsula_Resource?SWUDNSAI=31&xinfo=14-61856080-0%202NNN%20RT%281778262213322%2073%29%20q%280%20-1%20-1%20-1%29%20r%280%20-1%29%20B12%284%2c316%2c0%29&incident_id=686000620208051366-341139966646947726&edet=12&cinfo=04000000&rpinfo=0&cts=7D7M38hDi863PXV2J88apuTAArqpzBS1yDIp7MwAq4TqvNRQCPJq2erG7mpZ6FhU&cip=193.0.108.47&mth=GET" frameborder=0 width="100%" height="100%" marginheight="0px" marginwidth="0px">Request unsuccessful. Incapsula incident ID: 686000620208051366-3411

As we can see, website redirects us to its antirobot script, so it is not possible to access the content with simple requests.get. Maybe we can adjust headers of request and try once again?

In [24]:
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/147.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
}
response = requests.get(url, headers = headers)

print(response.text)

<html style="height:100%"><head><META NAME="ROBOTS" CONTENT="NOINDEX, NOFOLLOW"><meta name="format-detection" content="telephone=no"><meta name="viewport" content="initial-scale=1.0"><meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1"><script src="/Of-slaine-but-secrewes-the-Lye-my-lean-an-of-the" async></script><script type="text/javascript">if (sessionStorage) { sessionStorage.setItem('distil_referrer', document.referrer); }</script></head><body style="margin:0px;height:100%"><iframe id="main-iframe" src="/_Incapsula_Resource?SWUDNSAI=31&xinfo=17-85316629-0%202NNN%20RT%281778262213426%2022%29%20q%280%20-1%20-1%20-1%29%20r%280%20-1%29%20B12%284%2c316%2c0%29&incident_id=686000620208051366-475362114168295313&edet=12&cinfo=04000000&rpinfo=0&cts=OvWcEPxL03m9c76StdbAoTPaNt%2bK8zLKh6HlGOgafmj2OmPAlxdZpiLn8Uh90Y6R&cip=193.0.108.47&mth=GET" frameborder=0 width="100%" height="100%" marginheight="0px" marginwidth="0px">Request unsuccessful. Incapsula incident ID: 686000620208051366-47

It seems that simple requests library does not set us up for success in this matter, thus a sway towards selenium might be of help here.

Setting up.

In [27]:
chromepath = ChromeDriverManager().install()
service = Service(executable_path = chromepath)
options = webdriver.ChromeOptions()
driver = webdriver.Chrome(service = service, options = options)
driver.maximize_window()
driver.get(url)

Let's display the site's content and check if it works now. After switching to display the cell's output as a scrollable element and further examination, it is apparent that selenium has succeeded. 

In [28]:
print(driver.page_source)


<html lang="pl-PL"><head><script type="text/javascript">!function(){try{if("undefined"!=typeof sessionStorage){var e=sessionStorage.getItem("distil_referrer");(e||e==="")&&(Object.defineProperty(document,"referrer",{get:function(){return e}}),sessionStorage.removeItem("distil_referrer"))}}catch(e){}}();</script><script src="/Of-slaine-but-secrewes-the-Lye-my-lean-an-of-the" async=""></script><script type="text/javascript">!function(){try{if("undefined"!=typeof sessionStorage){var e=sessionStorage.getItem("distil_referrer");(e||e==="")&&(Object.defineProperty(document,"referrer",{get:function(){return e}}),sessionStorage.removeItem("distil_referrer"))}}catch(e){}}();</script>
    <meta charset="UTF-8">
    <meta http-equiv="x-ua-compatible" content="ie=edge">
    <meta name="viewport" content="width=device-width, initial-scale=1, shrink-to-fit=no">
    <meta name="mobile-web-app-capable" content="yes">
    <meta name="apple-mobile-web-app-capable" content="yes">
    <meta name="theme-co

Now let's deal with accepting cookies.

One of the challenges is that the webpage of news' archive of nbp.pl displays only 10 most recent articles, so throughout the scraping process and upon checking the content of &lt;article class="entry"&gt; it is required that after checking the scraper proceeds to the next page of search results.